# Paper-Ready Metric Plots

PSNR / SSIM vs transmission budget. Error bars = 95% CI of the mean: `mean ± 1.96 × (σ/√n)`.

**Run from `dlapisgs-utility/`:**
```bash
jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=120 \
    plotting/paper_plot_metrics.ipynb
```

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use("Agg")

In [2]:
# ------------------------------- setting start ------------------------------ #
color_palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
errorbar_color = "#3A3A3A"

# font
csfont = {'family': 'Times New Roman', 'serif': 'Times', 'size': 23}

# errorbar plot size
err_lw       = 1.5
err_capsize  = 4
err_capthick = 1.5

# figure size
figsize = (6.4, 4.8)

# set theme first, then rc — so seaborn doesn't clobber the font size
sns.set_theme(style="ticks", font="Times New Roman")
plt.rc('text', usetex=True)
plt.rc('font', **csfont)
# -------------------------------- setting end ------------------------------- #

In [ ]:
# ── I/O ──────────────────────────────────────────────────────────────────────
# Paths relative to dlapisgs-utility/ (set by the working-directory cell below)
SUMMARY_CSV = "output/0603/exp1_1600_vol_modes/summary_all.csv"
OUT_DIR     = "plotting/paper"

# CSV column to group lines by
GROUP_BY = "weight_mode"

# Keys to exclude entirely (e.g. implementation mistakes)
EXCLUDE_KEYS = ["det_gamma_over_d2"]

KEY_CONFIG = {
    # weight modes
    "volume":           {"label": "Volume (view-indep.)",         "marker": "^", "color": color_palette[2]},
    "volume_over_d2":   {"label": r"Vol/d$^2$ (view-dep.)",       "marker": "D", "color": color_palette[3]},
    "screen_area":      {"label": "Screen Area",                   "marker": "o", "color": color_palette[4]},
    "random":           {"label": "Random (control)",              "marker": "x", "color": color_palette[7]},
    # utility schemes
    "vd_lod":           {"label": "VD+LOD (baseline)",             "marker": "s", "color": color_palette[0]},
    "vd_lod_w":         {"label": "VD+LOD+W",                      "marker": "^", "color": color_palette[1]},
    "vd_lod_c":         {"label": "VD+LOD+C",                      "marker": "D", "color": color_palette[2]},
    "vd_lod_w_c":       {"label": "VD+LOD+W+C",                    "marker": "o", "color": color_palette[3]},
    "ml":               {"label": "ML (ours)",                     "marker": "P", "color": color_palette[5]},
    "oracle_loo":       {"label": "Oracle LOO (upper bound)",       "marker": "*", "color": color_palette[9]},
    "oracle_aoi":       {"label": "Oracle AOI",                    "marker": "h", "color": color_palette[8]},
    "oracle_combined":  {"label": "Oracle Combined",               "marker": "H", "color": color_palette[6]},
    # packing modes
    "tile_partial":     {"label": "Tile-partial (proposed)",       "marker": "o", "color": color_palette[0]},
    "tile_strict":      {"label": "Tile-strict",                   "marker": "s", "color": color_palette[1]},
    "progressive":      {"label": "Progressive (tile-agnostic)",   "marker": "^", "color": color_palette[2]},
}

KEY_ORDER = {
    "weight_mode":  ["screen_area", "volume_over_d2", "volume", "random"],
    "scheme":       ["vd_lod", "vd_lod_w", "ml", "oracle_loo"],
    "packing_mode": ["progressive", "tile_partial", "tile_strict"],
}

DPI = 300

In [4]:
import os

# cd to dlapisgs-utility/ regardless of where nbconvert was invoked
_cwd = Path(os.getcwd())
_root = None
for _candidate in [_cwd, _cwd.parent]:
    if (_candidate / "utility_calculation.py").exists():
        _root = _candidate
        break
if _root is None:
    try:
        _root = Path(__file__).resolve().parent.parent
    except NameError:
        _root = _cwd
os.chdir(_root)
print(f"cwd: {Path.cwd()}")

cwd: /mnt/data1/samk/gs-quic/cs5262_tile_quic/dlapisgs-utility


In [5]:
summary_csv = Path(SUMMARY_CSV)
if not summary_csv.exists():
    raise FileNotFoundError(f"{summary_csv}  (cwd={Path.cwd()})")

df = pd.read_csv(summary_csv)
df = df[~df[GROUP_BY].isin(EXCLUDE_KEYS)].copy()
print(f"Loaded {len(df)} rows | groups: {sorted(df[GROUP_BY].unique())} | budgets: {sorted(df['budget_mb'].unique())}")

Loaded 800 rows | groups: ['screen_area', 'volume', 'volume_over_d2'] | budgets: [np.float64(141.88218841552734), np.float64(1418.8218841552734)]


In [6]:
def aggregate_ci95(df, group_by):
    """mean ± 1.96·σ/√n per (group, budget, metric)"""
    records = []
    for key, gdf in df.groupby(group_by):
        for budget, bdf in gdf.groupby("budget_mb"):
            for metric in ("psnr", "ssim"):
                v = bdf[metric].dropna().values
                records.append({
                    group_by: key, "budget_mb": float(budget), "metric": metric,
                    "mean": v.mean(), "ci95": 1.96 * v.std(ddof=1) / np.sqrt(len(v)), "n": len(v),
                })
    return pd.DataFrame(records)

agg = aggregate_ci95(df, GROUP_BY)
print(agg.head(9).to_string(index=False))

   weight_mode   budget_mb metric      mean     ci95   n
   screen_area  141.882188   psnr 17.861596 1.281442 200
   screen_area  141.882188   ssim  0.593130 0.044088 200
   screen_area 1418.821884   psnr       inf      NaN 200
   screen_area 1418.821884   ssim  0.999632 0.000060 200
        volume  141.882188   psnr 22.471933 0.951203 100
        volume  141.882188   ssim  0.836998 0.026746 100
        volume 1418.821884   psnr 59.728386 0.166377 100
        volume 1418.821884   ssim  0.999265 0.000065 100
volume_over_d2  141.882188   psnr 24.108829 1.020382 100


/mnt/data1/samk/anaconda3/envs/gsquic/lib/python3.9/site-packages/numpy/_core/_methods.py:185: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


In [7]:
def resolve_order(keys):
    preferred = KEY_ORDER.get(GROUP_BY, [])
    ordered   = [k for k in preferred if k in keys]
    return ordered + [k for k in sorted(keys) if k not in ordered]


def plot_metric(agg, metric, ylabel, out_stem, out_dir):
    sub  = agg[agg["metric"] == metric]
    keys = resolve_order(sub[GROUP_BY].unique().tolist())

    fallback_markers = ["s", "^", "D", "o", "v", "P", "X"]

    fig, ax = plt.subplots(figsize=figsize)

    for i, key in enumerate(keys):
        cfg    = KEY_CONFIG.get(key, {})
        label  = cfg.get("label",  key)
        marker = cfg.get("marker", fallback_markers[i % len(fallback_markers)])
        color  = cfg.get("color",  color_palette[i % len(color_palette)])

        kdf = sub[sub[GROUP_BY] == key].sort_values("budget_mb")
        ax.errorbar(
            kdf["budget_mb"].values, kdf["mean"].values, yerr=kdf["ci95"].values,
            marker=marker, color=color, linewidth=2, markersize=8,
            capsize=err_capsize, elinewidth=err_lw, capthick=err_capthick,
            label=label,
        )

    ax.set_xlabel(r"Budget (MiB)")
    ax.set_ylabel(ylabel)
    ax.legend(loc="upper left", framealpha=0.9)

    # full box, ticks only on left and bottom
    for spine in ax.spines.values():
        spine.set_visible(True)
    ax.tick_params(direction="out", which="both", top=False, right=False)

    fig.tight_layout()
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / f"{out_stem}.png", dpi=DPI, bbox_inches="tight")
    fig.savefig(out_dir / f"{out_stem}.eps", format="eps", bbox_inches="tight")
    print(f"Wrote {out_dir}/{out_stem}.{{png,eps}}")
    plt.close(fig)

In [8]:
out_dir = Path(OUT_DIR)
plot_metric(agg, "psnr", r"Quality in PSNR (dB)", "psnr_vs_budget", out_dir)
plot_metric(agg, "ssim", r"Quality in SSIM",       "ssim_vs_budget", out_dir)
print("Done.")

The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote /tmp/nb_smoke2/psnr_vs_budget.{png,eps}


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Wrote /tmp/nb_smoke2/ssim_vs_budget.{png,eps}
Done.
